# Prediction Module - Homogeneous Embedding

    The main goal of the prediction module is to use the PropaPhenKG+ and the observations found in the detection module to cluster the observations into similar phenomenon clusters by performing homogeneous embeddings

In [1]:
%load_ext autoreload
%autoreload 2

## Libraries

### Installing

In [2]:
#!pip install pandas
#!pip install tqdm
#!pip install -U scikit-learn
#!pip install matplotlib
#!pip install dgl

### Standard

In [3]:
import dgl

In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import glob
import pykeen
import torch

### Custom libraries

## Globals

In [5]:
path_to_kb_gazetteer = "../Detection/data/gazetteers/kbgazetteer.csv"
path_to_netwoork_gazetteer = "../Detection/data/gazetteers/world_gazetteer.csv"
path_to_observationcsv = "../Detection/data/csv/observations_phrase.csv"

In [6]:
path_to_covid_journalobservationcsv = "../data/neo4j/covid_observations_journal.csv"
path_to_covid_medicalobservationcsv = "../data/neo4j/covid_observations_medical.csv"
path_to_covid_socialobservationcsv = "../data/neo4j/covid_observations_social.csv"
path_to_monkeypox_journalobservationcsv = "../data/neo4j/monkeypox_observations_journal.csv"
path_to_monkeypox_medicalobservationcsv = "../data/neo4j/monkeypox_observations_medical.csv"
path_to_monkeypox_socialobservationcsv = "../data/neo4j/monkeypox_observations_social.csv"

## Observation Embedding

    It should get the observations and the PropaPhenKG+ to transform the observations into observation vectors

### Step 1: Prepare Your Data with Pandas
Assume you have two CSV files:

    Nodes CSV: Contains node IDs, types, and potentially some features.
    Edges CSV: Contains source node IDs, destination node IDs, and edge types.

### Step 2: Load the Data with PandasLoad the node and edge data using Pandas:Load the node and edge data using Pandas:
Load the node and edge data using Pandas:

### Step 3: Create the Heterogeneous Graph with DGL

Next, create the heterogeneous graph using DGL's heterograph function:

In [7]:
import pandas as pd
import dgl
import torch

# Paths to your node and edge CSV files
path_to_worldkg_nodes = "../data/worldkg/worldkg_nodes.csv"
path_to_worldkg_edges = "../data/worldkg/worldkg_edges.csv"
# Define the chunk size
chunk_size = 100000  # Adjust based on your memory limits

In [8]:
# Initialize dictionaries to hold nodes, edges, and mappings
node_type_dict = {}
edge_data = {}
node_id_to_type = {}
id_remap = {}
max_remapped_ids = {}
current_id = {}
unmapped_ids = set()

In [9]:
# Step 1: Load and process nodes in chunks
node_chunks = pd.read_csv(path_to_worldkg_nodes, chunksize=chunk_size)
# Step 2: Load and process edges in chunks
edge_chunks = pd.read_csv(path_to_worldkg_edges, chunksize=chunk_size)

In [10]:
# Node chunks
for chunk in node_chunks:
    # Rename columns
    chunk = chunk.rename(columns={'id:ID': 'node_id', ':LABEL': 'node_type'})
    
    # Remove the 'wkg:' prefix from node_id
    chunk['node_id'] = chunk['node_id'].str.replace('wkg:', '', regex=False).astype(int)
    
    # Process the chunk and create a new ID mapping
    for node_type, group in chunk.groupby('node_type'):
        if node_type not in node_type_dict:
            node_type_dict[node_type] = []
            max_remapped_ids[node_type] = -1
            current_id[node_type] = 0  # Initialize current_id for each node type
        
        # Remap node IDs
        for old_id in group['node_id']:
            if old_id not in id_remap:
                id_remap[old_id] = current_id[node_type]
                current_id[node_type] += 1
            remapped_id = id_remap[old_id]
            node_type_dict[node_type].append(remapped_id)
            max_remapped_ids[node_type] = max(max_remapped_ids[node_type], remapped_id)

        # Create a mapping from node ID to node type
        for node_id in group['node_id']:
            node_id_to_type[node_id] = node_type

/tmp/ipykernel_1588863/1901994379.py:2: DtypeWarning: Columns (5,12,17,21,23,25,26,33,37,38,43,44,47,48,51,54,57,59,61,66,68,71,72,73,78,79,81,83,92,96,99,102,112,114,117,125,126,129,130,131,132,138,141,146,149,152,154,158,160,169,183,186,191,198,199,200,202,204,208,211,214,216,218,224,230,232,236,241,242,244,245,249,252,254,255,257,263,273,283,284,285,287,289,290,291) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in node_chunks:
/tmp/ipykernel_1588863/1901994379.py:2: DtypeWarning: Columns (12,17,21,23,24,25,33,37,38,39,40,43,44,46,51,54,57,61,66,68,69,72,73,76,77,78,79,81,82,83,86,90,92,95,96,99,102,111,112,114,117,126,129,130,131,132,138,141,142,149,151,152,159,160,164,169,176,178,179,180,183,185,198,199,202,204,208,211,212,217,221,224,235,236,241,245,246,248,249,251,252,254,256,257,259,261,264,273,280,283,285,287,288,289,290,291,293,294) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in node_chunks:
/tmp

In [11]:
max(list(id_remap.values()))

1241529

In [12]:
max_id = 0
for old_id in id_remap.keys():
    if node_id_to_type[old_id] == 'City' and id_remap[old_id] == 820717:
        max_id = id_remap[old_id]
print(max_id)

0


In [13]:
group.iloc[0]

node_type                        Village
Name                    "Great Wishford"
wkgs_refSePtsPostort                 NaN
wkgs_state                           NaN
wkgs_borderType                      NaN
                              ...       
wkgs_nameEn                          NaN
wkgs_addrProvince                    NaN
wkgs_officialNameRu                  NaN
wkgs_sortingName                     NaN
wkgs_namePrefix                      NaN
Name: 1200000, Length: 295, dtype: object

In [14]:
group.iloc[0].tolist()

['Village',
 '"Great Wishford"',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 'wkg:geo271443',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 271443,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan

In [15]:
node_id_to_type[473711478]

'Village'

In [16]:
max_remapped_ids

{'City': 10090,
 'County': 7034,
 'Island': 31225,
 'Municipality': 1193,
 'Region': 1451,
 'State': 1757,
 'Village': 1241529,
 'Continent': 6,
 'Country': 215}

In [17]:
id_remap[473711478]

839277

In [18]:
max_remapped_ids

{'City': 10090,
 'County': 7034,
 'Island': 31225,
 'Municipality': 1193,
 'Region': 1451,
 'State': 1757,
 'Village': 1241529,
 'Continent': 6,
 'Country': 215}

In [19]:
for chunk in edge_chunks:
    # Rename columns
    chunk = chunk.rename(columns={':START_ID': 'src', ':END_ID': 'dst', ':TYPE': 'edge_type'})
    
    # Remove the 'wkg:' prefix from src and dst
    chunk['src'] = chunk['src'].str.replace('wkg:', '', regex=False).astype(int)
    chunk['dst'] = chunk['dst'].str.replace('wkg:', '', regex=False).astype(int)
    
    # Remap the edges to use the new IDs
    chunk['src_mapped'] = chunk['src'].map(id_remap)
    chunk['dst_mapped'] = chunk['dst'].map(id_remap)

    # Filter out any edges with unmapped or out-of-range IDs
    valid_edges = chunk.dropna(subset=['src_mapped', 'dst_mapped'])

    # Process each row individually to handle different node types
    for idx, row in valid_edges.iterrows():
        src_type = node_id_to_type.get(row['src'], None)
        dst_type = node_id_to_type.get(row['dst'], None)
        etype = row['edge_type']

        if src_type is None or dst_type is None:
            continue  # Skip this edge if any node type is missing

        key = (src_type, etype, dst_type)
        if key not in edge_data:
            edge_data[key] = ([], [])
        
        # Append the current edge's source and destination
        edge_data[key][0].append(row['src_mapped'])
        edge_data[key][1].append(row['dst_mapped'])


In [20]:
for key in edge_data.keys():
    src_value,dst_value = edge_data[key]
    if 'City' == key[0] and max(src_value) > 10091:
        print(max(src_value))
    if 'City' == key[2] and max(dst_value) > 10091:
        print(max(dst_value))

In [21]:
max_remapped_ids

{'City': 10090,
 'County': 7034,
 'Island': 31225,
 'Municipality': 1193,
 'Region': 1451,
 'State': 1757,
 'Village': 1241529,
 'Continent': 6,
 'Country': 215}

In [22]:
for key in edge_data.keys():
    src_type = key[0]
    dst_type = key[2]
    

In [23]:
# Step 3: Convert lists to tensors in edge_data
edge_data2 = {}
for key in edge_data:
    src_list, dst_list = edge_data[key]
    edge_data2[key] = (torch.tensor(src_list, dtype=torch.int64), torch.tensor(dst_list, dtype=torch.int64))

# Step 4: Create the heterogeneous graph using max_remapped_ids + 1 for num_nodes_dict
num_nodes_dict = {ntype: max_remapped_ids[ntype] + 1 for ntype in max_remapped_ids}

hg = dgl.heterograph(edge_data2, num_nodes_dict=num_nodes_dict)

### Step 6: Define a Model for the Heterogeneous Graph

Given that you've created a heterogeneous graph, you can now define a model to learn from this graph. Common models for heterogeneous graphs include HeteroRGCN (Heterogeneous Relational Graph Convolutional Network) and HeteroGraphSAGE.

import torch
import torch.nn as nn
from dgl.nn import RelGraphConv

class HeteroRGCN(nn.Module):
    def __init__(self, hidden_feats, out_feats, rel_names):
        super(HeteroRGCN, self).__init__()
        self.hidden_feats = hidden_feats
        self.out_feats = out_feats
        self.rel_names = rel_names
        self.conv1 = RelGraphConv(hidden_feats, hidden_feats, len(rel_names), "basis")
        self.conv2 = RelGraphConv(hidden_feats, out_feats, len(rel_names), "basis")

    def forward(self, g, inputs):
        h = self.conv1(g, inputs)
        h = {k: torch.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        return h

    def init_embeddings(self, g, embedding_dim):
        embeddings = {}
        for ntype in g.ntypes:
            num_nodes = g.number_of_nodes(ntype)
            embeddings[ntype] = nn.Embedding(num_nodes, embedding_dim)
        return embeddings

### Step 7: Initialize Node Embeddings

Since you might not have node features, you can initialize embeddings for each node type:

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn.pytorch as dglnn

# Step 1: Define the HeteroRGCN model
class HeteroRGCN(nn.Module):
    def __init__(self, num_node_types, in_feats, hidden_feats, out_feats, etypes):
        super(HeteroRGCN, self).__init__()
        self.rel_graph_convs = nn.ModuleDict()
        
        # Create a GraphConv for each edge type and use string keys for ModuleDict
        for srctype, etype, dsttype in etypes:
            key = f'{srctype}_{etype}_{dsttype}'  # Convert the tuple to a string key
            self.rel_graph_convs[key] = dglnn.GraphConv(in_feats, hidden_feats, norm='right', weight=True, bias=True, allow_zero_in_degree=True)
        
        # Define a hidden-to-output layer for each node type
        self.out_layer = nn.ModuleDict({
            ntype: nn.Linear(hidden_feats, out_feats) for ntype in num_node_types
        })
        
    def forward(self, graph, inputs):
        h_dict = {ntype: inputs.get(ntype, None) for ntype in graph.ntypes}  # Initialize h_dict with inputs
        
        # Apply GraphConv for each edge type using the string keys
        for srctype, etype, dsttype in graph.canonical_etypes:
            key = f'{srctype}_{etype}_{dsttype}'  # Convert the tuple to a string key
            if srctype in inputs and h_dict[srctype] is not None:
                h = self.rel_graph_convs[key](graph[(srctype, etype, dsttype)], h_dict[srctype])
                if dsttype not in h_dict:
                    h_dict[dsttype] = h
                else:
                    h_dict[dsttype] += h
        
        # Apply non-linearity (ReLU)
        h_dict = {k: F.relu(h) for k, h in h_dict.items() if h is not None}
        
        # Apply hidden-to-output layer for each node type
        h_dict = {k: self.out_layer[k](h) for k, h in h_dict.items() if h is not None}
        
        return h_dict

# Step 2: Prepare the graph, features, and labels
# Set feature and output sizes
in_feats = 2  # Set input feature size to 2
hidden_feats = 2  # Set hidden feature size to 2
out_feats = 2  # Set output feature size to 2

# Assuming you have already created 'hg' (heterogeneous graph)
print("Node types in the graph:", hg.ntypes)
print("Canonical edge types in the graph:", hg.canonical_etypes)

# Initialize node features with random data for demonstration purposes
features = {ntype: torch.randn(hg.num_nodes(ntype), in_feats) for ntype in hg.ntypes}
print("Features dictionary keys:", features.keys())

# Initialize labels with random data for demonstration purposes
labels = {ntype: torch.randint(0, 5, (hg.num_nodes(ntype),)) for ntype in hg.ntypes}  # Example with 5 classes

# Automatically determine the number of classes
num_classes = max([labels[ntype].max().item() for ntype in labels]) + 1
print("Number of classes detected:", num_classes)

# Step 3: Add self-loops only for non-bipartite edge types
for srctype, etype, dsttype in hg.canonical_etypes:
    if srctype == dsttype:
        hg = dgl.add_self_loop(hg, etype=(srctype, etype, dsttype))

# Step 4: Initialize the model and optimizer
model = HeteroRGCN(hg.ntypes, in_feats, hidden_feats, out_feats, hg.canonical_etypes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Step 5: Define the loss function
def compute_loss(logits, labels):
    """
    Compute the loss for node classification.
    
    Parameters:
    - logits: Dictionary of output logits from the model for each node type.
    - labels: Dictionary of ground truth labels for each node type.
    
    Returns:
    - loss: Computed loss value.
    """
    loss = 0
    for ntype in logits:
        if ntype in labels:
            loss += F.cross_entropy(logits[ntype], labels[ntype])
    return loss

# Step 6: Training loop with manual mini-batching
num_epochs = 5  # Number of training epochs
batch_size = 2  # Adjust based on your memory constraints

for epoch in range(num_epochs):
    model.train()
    for ntype in hg.ntypes:
        num_batches = (hg.num_nodes(ntype) + batch_size - 1) // batch_size
        
        for i in range(num_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, hg.num_nodes(ntype))
            
            # Select the mini-batch of nodes for this type
            batch_nodes = torch.arange(start_idx, end_idx)
            
            # Create subgraph with only the batch nodes of this type
            subgraph = dgl.node_subgraph(hg, {ntype: batch_nodes})
            
            # Select corresponding features and labels
            batch_features = {ntype: features[ntype][start_idx:end_idx]}
            batch_labels = {ntype: labels[ntype][start_idx:end_idx]}
            
            # Forward pass
            logits = model(subgraph, batch_features)
            
            # Compute loss
            loss = compute_loss(logits, batch_labels)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f'Epoch {epoch}, Batch {i}/{num_batches}, Loss: {loss.item()}')

# Step 7: Inference
# After training, you can perform inference on the entire graph or on specific batches of nodes.
model.eval()
with torch.no_grad():
    logits = model(hg, features)
    # Further processing of logits can be done as per your requirement


Node types in the graph: ['City', 'Continent', 'Country', 'County', 'Island', 'Municipality', 'Region', 'State', 'Village']
Canonical edge types in the graph: [('City', 'wdp:P10254', 'Village'), ('City', 'wdp:P103', 'Village'), ('City', 'wdp:P105', 'Village'), ('City', 'wdp:P1056', 'Village'), ('City', 'wdp:P106', 'Village'), ('City', 'wdp:P108', 'Village'), ('City', 'wdp:P10888', 'Village'), ('City', 'wdp:P112', 'Island'), ('City', 'wdp:P112', 'Village'), ('City', 'wdp:P119', 'City'), ('City', 'wdp:P1196', 'Village'), ('City', 'wdp:P1303', 'Village'), ('City', 'wdp:P131', 'City'), ('City', 'wdp:P131', 'Country'), ('City', 'wdp:P131', 'County'), ('City', 'wdp:P131', 'Island'), ('City', 'wdp:P131', 'Municipality'), ('City', 'wdp:P131', 'Region'), ('City', 'wdp:P131', 'State'), ('City', 'wdp:P131', 'Village'), ('City', 'wdp:P1313', 'Village'), ('City', 'wdp:P1336', 'Country'), ('City', 'wdp:P1343', 'Village'), ('City', 'wdp:P136', 'Island'), ('City', 'wdp:P136', 'Village'), ('City', 'wdp

TypeError: unsupported operand type(s) for +=: 'NoneType' and 'Tensor'